In [1]:
from datasets import load_dataset
import pandas as pd
import json

In [3]:
from datasets import load_dataset
import pandas as pd
import json

print("Downloading Criteo dataset from HuggingFace...")

dataset = load_dataset(
    "criteo/criteo-attribution-dataset",
    split="train[:100000]"
)

df_raw = dataset.to_pandas()

print("Shape:", df_raw.shape)
print("\nColumns:", df_raw.columns.tolist())
print("\nFirst 3 rows:")
print(df_raw.head(3))
print("\nMissing values:")
print(df_raw.isnull().sum())

README.md: 0.00B [00:00, ?B/s]

criteo_attribution_dataset.tsv.gz:   0%|          | 0.00/653M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16468027 [00:00<?, ? examples/s]

Shape: (100000, 22)

Columns: ['timestamp', 'uid', 'campaign', 'conversion', 'conversion_timestamp', 'conversion_id', 'attribution', 'click', 'click_pos', 'click_nb', 'cost', 'cpo', 'time_since_last_click', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cat9']

First 3 rows:
   timestamp       uid  campaign  conversion  conversion_timestamp  \
0          0  20073966  22589171           0                    -1   
1          2  24607497    884761           0                    -1   
2          2  28474333  18975823           0                    -1   

   conversion_id  attribution  click  click_pos  click_nb  ...  \
0             -1            0      0         -1        -1  ...   
1             -1            0      0         -1        -1  ...   
2             -1            0      0         -1        -1  ...   

   time_since_last_click      cat1     cat2      cat3      cat4      cat5  \
0                     -1   5824233  9312274   3490278  29196072  11409686   
1     

In [4]:
print("Shape:", df_raw.shape)
print("\nColumns:", df_raw.columns.tolist())

Shape: (100000, 22)

Columns: ['timestamp', 'uid', 'campaign', 'conversion', 'conversion_timestamp', 'conversion_id', 'attribution', 'click', 'click_pos', 'click_nb', 'cost', 'cpo', 'time_since_last_click', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cat9']


In [5]:
print("\nFirst few sample rows:")
print(df_raw.head())


First few sample rows:
   timestamp       uid  campaign  conversion  conversion_timestamp  \
0          0  20073966  22589171           0                    -1   
1          2  24607497    884761           0                    -1   
2          2  28474333  18975823           0                    -1   
3          3   7306395  29427842           1               1449193   
4          3  25357769  13365547           0                    -1   

   conversion_id  attribution  click  click_pos  click_nb  ...  \
0             -1            0      0         -1        -1  ...   
1             -1            0      0         -1        -1  ...   
2             -1            0      0         -1        -1  ...   
3        3063962            0      1          0         7  ...   
4             -1            0      0         -1        -1  ...   

   time_since_last_click      cat1      cat2      cat3      cat4      cat5  \
0                     -1   5824233   9312274   3490278  29196072  11409686   
1 

In [6]:
print("\nMissing values:")
print(df_raw.isnull().sum())


Missing values:
timestamp                0
uid                      0
campaign                 0
conversion               0
conversion_timestamp     0
conversion_id            0
attribution              0
click                    0
click_pos                0
click_nb                 0
cost                     0
cpo                      0
time_since_last_click    0
cat1                     0
cat2                     0
cat3                     0
cat4                     0
cat5                     0
cat6                     0
cat7                     0
cat8                     0
cat9                     0
dtype: int64


In [9]:
#creating path for saving file
import os
os.makedirs('../data/raw', exist_ok=True)
print("Folder created:", os.path.abspath('../data/raw'))

Folder created: /Users/amber/Documents/data/raw


In [10]:
#converting and saving file as JSON
records = df_raw.to_dict(orient='records')

with open('../data/raw/criteo_sample.json', 'w') as f:
    json.dump(records, f)

print(f"Saved {len(records):,} records to data/raw/criteo_sample.json")

Saved 100,000 records to data/raw/criteo_sample.json


In [12]:
#load data to mongo
from pymongo import MongoClient
import json

# Connect to MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['ecommerce_db']
collection = db['criteo_raw']

# Clear collection if re-running
collection.drop()

# Load JSON and insert into MongoDB
with open('../data/raw/criteo_sample.json', 'r') as f:
    records = json.load(f)

# Insert in batches of 10,000
batch_size = 10000
for i in range(0, len(records), batch_size):
    batch = records[i:i+batch_size]
    collection.insert_many(batch)
    print(f"Inserted {min(i+batch_size, len(records)):,} / {len(records):,}")

total = collection.count_documents({})
print(f"\nMongoDB confirmed: {total:,} documents in criteo_raw")

sample = collection.find_one()
print("\nSample document:")
print(sample)

Inserted 10,000 / 100,000
Inserted 20,000 / 100,000
Inserted 30,000 / 100,000
Inserted 40,000 / 100,000
Inserted 50,000 / 100,000
Inserted 60,000 / 100,000
Inserted 70,000 / 100,000
Inserted 80,000 / 100,000
Inserted 90,000 / 100,000
Inserted 100,000 / 100,000

MongoDB confirmed: 100,000 documents in criteo_raw

Sample document:
{'_id': ObjectId('69ec9a7d4dfcc876fb46add8'), 'timestamp': 0, 'uid': 20073966, 'campaign': 22589171, 'conversion': 0, 'conversion_timestamp': -1, 'conversion_id': -1, 'attribution': 0, 'click': 0, 'click_pos': -1, 'click_nb': -1, 'cost': 1e-05, 'cpo': 0.390793596019, 'time_since_last_click': -1, 'cat1': 5824233, 'cat2': 9312274, 'cat3': 3490278, 'cat4': 29196072, 'cat5': 11409686, 'cat6': 1973606, 'cat7': 25162884, 'cat8': 29196072, 'cat9': 29196072}
